# Highbay schema/prose fine-tune - v5

Objective: Fine-tune Qwen2.5-1.5B-Instruct for Typed Markdown IR & AST Generation (v5).

In [ ]:
# 1. Mount Google Drive & Create Missing Training Directories
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/HighbayGeniusTraining"
TRAINING_OUTPUT_DIR = os.path.join(DRIVE_BASE, "training", "outputs")

os.makedirs(TRAINING_OUTPUT_DIR, exist_ok=True)

In [ ]:
# 2. Setup & Installation
!pip install -q torch transformers datasets unsloth trl gguf

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
# 3. Configuration Parameters
BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
RUN_ID       = "v5-qwen2.5-1.5b"
SEED         = 3407
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 16
EVAL_FRACTION = 0.2
TARGET       = "ast"
MAX_SEQ_LENGTH = 2048

In [ ]:
# 4. Model & LoRA Initialization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
# 5. Dataset Loading & Formatting
REPO_URL = "https://github.com/tmzt/TrainingExperiments.git"
CLONE_DIR = "/content/TrainingExperiments"

LOCAL_DATASET_PATH = "inputs/synthetic_typed_markdown_v5.jsonl"
CLONED_DATASET_PATH = os.path.join(CLONE_DIR, "Highbay/Local/v5/inputs/synthetic_typed_markdown_v5.jsonl")
DRIVE_DATASET_PATH = os.path.join(DRIVE_BASE, "datasets", "processed", "synthetic_typed_markdown_v5.jsonl")

# If local path doesn't exist and we're not in the repo, try cloning the repo (Colab flow)
if not os.path.exists(LOCAL_DATASET_PATH) and not os.path.exists(CLONED_DATASET_PATH):
    print("Local dataset not found. Cloning TrainingExperiments repository...")
    try:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    except Exception as e:
        print(f"Failed to clone repository: {e}")

# Resolve which dataset path to load
if os.path.exists(LOCAL_DATASET_PATH):
    DATASET_PATH = LOCAL_DATASET_PATH
    print(f"Reading dataset from local repository: {DATASET_PATH}")
elif os.path.exists(CLONED_DATASET_PATH):
    DATASET_PATH = CLONED_DATASET_PATH
    print(f"Reading dataset from cloned repository: {DATASET_PATH}")
elif os.path.exists(DRIVE_DATASET_PATH):
    DATASET_PATH = DRIVE_DATASET_PATH
    print(f"Reading dataset from Google Drive: {DATASET_PATH}")
else:
    raise FileNotFoundError(f"Could not find dataset at '{LOCAL_DATASET_PATH}', '{CLONED_DATASET_PATH}', or '{DRIVE_DATASET_PATH}'")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.train_test_split(test_size=EVAL_FRACTION, seed=SEED)

PROMPT_TEMPLATE = """Below is an instruction that describes a UI design action or natural language prompt. Write the corresponding Typed Markdown Intermediate Representation (IR).

### Instruction:
{prompt}

### Typed Markdown IR:
{target_ir}"""

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    targets = examples["target_ir"]
    texts = []
    for p, t in zip(prompts, targets):
        text = PROMPT_TEMPLATE.format(prompt=p, target_ir=t) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 6. Training Arguments & Trainer Execution
output_run_dir = os.path.join(TRAINING_OUTPUT_DIR, f"outputs_{RUN_ID}")
os.makedirs(output_run_dir, exist_ok=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = output_run_dir,
        seed = SEED,
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 7. Save Adapter & Export GGUF
import shutil
import subprocess

# Output configurations
EXPORT_LORA_GGUF = True   # Standalone LoRA GGUF adapter (supported in updated wllama)
EXPORT_FULL_GGUF = False  # Merged full model GGUF (optional, disabled by default)

LOCAL_OUTPUT_DIR = f"/content/outputs_{RUN_ID}"
adapter_local_path = os.path.join(LOCAL_OUTPUT_DIR, f"adapter_{RUN_ID}")
os.makedirs(adapter_local_path, exist_ok=True)

# 7a. Save adapter model locally first (fast & secure)
print("Saving LoRA adapter locally...")
model.save_pretrained(adapter_local_path)
tokenizer.save_pretrained(adapter_local_path)

# 7b. Export Standalone LoRA GGUF (using llama.cpp conversion script)
lora_gguf_local = os.path.join(LOCAL_OUTPUT_DIR, f"lora_adapter_{RUN_ID}.gguf")
if EXPORT_LORA_GGUF:
    print("Exporting standalone LoRA GGUF...")
    llama_cpp_path = "/content/llama.cpp"
    if not os.path.exists(llama_cpp_path):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", llama_cpp_path], check=True)
    
    convert_script = os.path.join(llama_cpp_path, "convert_lora_to_gguf.py")
    try:
        subprocess.run([
            "python3", convert_script,
            "--base", BASE_MODEL,
            "--outfile", lora_gguf_local,
            adapter_local_path
        ], check=True)
        print(f"Standalone LoRA GGUF successfully created at: {lora_gguf_local}")
    except Exception as e:
        print(f"Failed to convert LoRA adapter to GGUF: {e}")

# 7c. Export Merged Full Model GGUF (Large file)
gguf_local_dir = os.path.join(LOCAL_OUTPUT_DIR, f"gguf_{RUN_ID}_full")
if EXPORT_FULL_GGUF:
    print("Exporting full merged GGUF model...")
    try:
        model.save_pretrained_gguf(gguf_local_dir, tokenizer, quantization_method="q4_k_m")
        print(f"Full GGUF model successfully created at: {gguf_local_dir}")
    except Exception as e:
        print(f"Full GGUF model export failed: {e}")

# 7d. Copy all local files to Google Drive
adapter_drive_path = os.path.join(TRAINING_OUTPUT_DIR, f"adapter_{RUN_ID}")
print(f"Copying adapter files to Google Drive: {adapter_drive_path}")
shutil.copytree(adapter_local_path, adapter_drive_path, dirs_exist_ok=True)

if EXPORT_LORA_GGUF and os.path.exists(lora_gguf_local):
    lora_gguf_drive = os.path.join(TRAINING_OUTPUT_DIR, f"lora_adapter_{RUN_ID}.gguf")
    print(f"Copying standalone LoRA GGUF to Google Drive: {lora_gguf_drive}")
    shutil.copy(lora_gguf_local, lora_gguf_drive)

if EXPORT_FULL_GGUF and os.path.exists(gguf_local_dir):
    gguf_drive_dir = os.path.join(TRAINING_OUTPUT_DIR, f"gguf_full_{RUN_ID}")
    print(f"Copying full merged GGUF model to Google Drive: {gguf_drive_dir}")
    shutil.copytree(gguf_local_dir, gguf_drive_dir, dirs_exist_ok=True)

print("All save and export operations finished successfully!")